# Pre-body

## Clearing past runs (optional)

In [1]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

## IIC-OSIC Env Setup

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [3]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-10-08 22:52:14,147 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-10-08 22:52:14,156 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


## Loading the project config

In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

22:52:14 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
22:52:14 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/logs/SymXplorer_2025-10-08_22-52-14.log
22:52:14 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
22:52:14 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: TwoPointsDE, type=nevergrad, budget=10, random_seed=48
22:52:14 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
22:52:14 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
22:52:14 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
22:52:14 - SymXplorer.domains: [INFO] 	Number of target specs: 3
22:52:14 - SymXplorer.domains: [INFO] 		- TargetSpec(name=ugf, target=200e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=100.0, enable=True, description=Unitiy gain frequency)
22:52

Project_Setup(name='5T-OTA', description='5 Transistor OTA example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2'), netlist=PosixPath('spice/ota-5t_tb-loopgain.spice'), outdir=PosixPath('sizing/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07)}), pvt=PVT(temp=25, corner='tt', supply=1.8), dut_params=[Param(name='x_dut_nfet_input_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False), Param(name='x_dut_nfet_input_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06)

## Create a SPICE simulator wrapper

In [5]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

22:52:14 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output
22:52:14 - SymXplorer.spicelib: [INFO] --------------------------------------------------
22:52:14 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
22:52:14 - SymXplorer.spicelib: [INFO] 	📝 Project: 5T-OTA
22:52:14 - SymXplorer.spicelib: [INFO] 	📜 Schematic: ota-5t_tb-loopgain
22:52:14 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output
22:52:14 - SymXplorer.spicelib: [INFO] --------------------------------------------------
22:52:14 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
22:52:14 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
22:52:14 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['v_dd', 'GND', 'v_ss', 'v_in', 'v_ena', 'vr1', 'net1', 'vf1', 'net2', 'net3', 'net4

## Create an optimizer object

In [6]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

22:52:14 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs


## Sanity Check

In [7]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

22:52:14 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
22:52:14 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
22:52:15 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output/sanity_check/5T-OTA_sanity.log
22:52:15 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output/sanity_check/5T-OTA_sanity.raw
22:52:15 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
22:52:15 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Main Body

## Optimization

In [8]:
circuit_optimizer.parameterize()

Dict(x_dut_nfet_input_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_input_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_mirror_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_mirror_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_pfet_load_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_pfet_load_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_input_w': 50.0, 'x_dut_nfet_input_l': 50.0, 'x_dut_nfet_mirror_w': 50.0, 'x_dut_nfet_mirror_l': 50.0, 'x_dut_pfet_load_w': 50.0, 'x_dut_pfet_load_l': 50.0}

In [9]:
_ = circuit_optimizer.optimize()

22:52:15 - SymXplorer.optimizer: [INFO] Optimization process started.
22:52:15 - SymXplorer.optimizer: [INFO] Optimizer is set to TwoPointsDE with budget = 10
Optimizing: 100%|██████████| 10/10 [00:05<00:00,  1.81trial/s]
22:52:21 - SymXplorer.optimizer: [INFO] Optimization process completed.


In [ ]:
circuit_optimizer.plot_score(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

22:52:21 - SymXplorer.plotter: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/loss_curve.html
22:52:21 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [11]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
# metadata

22:52:21 - SymXplorer.optimizer: [INFO] best loss: -84.56955245155288


In [12]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

x_dut_nfet_input_w: 4.55e-06
x_dut_nfet_input_l: 7.37e-06
x_dut_nfet_mirror_w: 4.70e-06
x_dut_nfet_mirror_l: 6.00e-06
x_dut_pfet_load_w: 3.49e-06
x_dut_pfet_load_l: 4.64e-06


In [13]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="out")

22:52:22 - SymXplorer.optimizer: [INFO] total score: -84.56955245155288
22:52:22 - SymXplorer.optimizer: [INFO] 	Spec 'ugf': curr_val=35520910.0, score=-64.83202042906248
22:52:22 - SymXplorer.optimizer: [INFO] 	Spec 'dcgain': curr_val=31.97276, score=0.0
22:52:22 - SymXplorer.optimizer: [INFO] 	Spec 'pm': curr_val=118.0, score=-19.7375320224904


### (3) Metric Trace

In [14]:
_ = circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='ugf', show=True)

22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [ ]:
circuit_optimizer.plot_score_value_by_spec(spec_name="dcgain", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="ugf", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="pm", show=True)

22:52:22 - SymXplorer.plotter: [INFO] 	min loss -1.5068859280010782; max loss 0.0
22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


22:52:22 - SymXplorer.plotter: [INFO] 	min loss -70.9326806031895; max loss -48.34578750610794
22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


22:52:22 - SymXplorer.plotter: [INFO] 	min loss -20.80297712989976; max loss 0.0
22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [16]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_pfet_load_w", param_y="x_dut_pfet_load_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_input_w", param_y="x_dut_nfet_input_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_mirror_w", param_y="x_dut_nfet_mirror_l", show=True)

22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


22:52:22 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


(tensor([2.9773e-06, 5.5636e-06, 4.6982e-06, 1.4772e-06, 5.9619e-06, 1.6155e-06,
         2.7638e-06, 1.7982e-06, 3.1857e-06, 8.9155e-06]),
 tensor([3.1159e-06, 3.6874e-06, 6.0008e-06, 5.4442e-07, 5.8263e-06, 7.9259e-06,
         7.7607e-06, 3.4971e-06, 1.9683e-06, 5.2638e-06]))

# Checkpointing

In [17]:
name = str(PROJECT_SETUP.ws_root/Path("sizing/data/checkpoint"))
circuit_optimizer.save_checkpoint(name=name)

22:52:22 - SymXplorer.optimizer: [INFO] ✅ Checkpoint saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/data/checkpoint.json


In [18]:
name

'/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/data/checkpoint'